# Lesson 3: Performing calculations

In [16]:
# Before you start, please run the following code to set up your environment.
# This code will reset the environment (if needed) and prepare the resources for the lesson.
# It does this by quickly running through all the code from the previous lessons.

!sh ./ro_shared_data/reset.sh
%run ./ro_shared_data/lesson_2_prep.py lesson3
%run ./ro_shared_data/lesson_3_prep.py lesson3

import os

agentId = os.environ['BEDROCK_AGENT_ID']
agentAliasId = os.environ['BEDROCK_AGENT_ALIAS_ID']
region_name = 'us-west-2'
lambda_function_arn = os.environ['LAMBDA_FUNCTION_ARN']
action_group_id = os.environ['ACTION_GROUP_ID']

Resetting environment (if nessesary)
Found: mugs-customer-support-agent
Deleting alias: AgentTestAlias (ID: TSTALIASID)
Error deleting alias AgentTestAlias: An error occurred (ValidationException) when calling the DeleteAgentAlias operation: 1 validation error detected: Value 'TSTALIASID' at 'agentAliasId' failed to satisfy constraint: Member must satisfy regular expression pattern: (?!\bTSTALIASID\b)[0-9a-zA-Z]+
Deleting alias: test (ID: XUZGDDMGFI)
Deletion initiated for alias: test
Alias test has been successfully deleted.
Deleting agent: mugs-customer-support-agent (ID: 7LFFPZNDWA)
Deletion initiated for agent: mugs-customer-support-agent
Waiting for agent mugs-customer-support-agent to be deleted...
Waiting for agent mugs-customer-support-agent to be deleted...
Agent mugs-customer-support-agent has been successfully deleted.
Agent reset process completed.
Deleting Lambda function: dlai-support-agent-WQJIW
Lambda function dlai-support-agent-WQJIW deleted successfully.
Lambda reset 

## Start of lesson

In [17]:
import boto3
import uuid
from helper import *

In [18]:
bedrock_agent = boto3.client(service_name='bedrock-agent', region_name=region_name)

In [19]:
update_agent_action_group_response = bedrock_agent.update_agent_action_group(
    actionGroupName='customer-support-actions',
    actionGroupState='ENABLED',
    actionGroupId=action_group_id,
    agentId=agentId,
    agentVersion='DRAFT',
    actionGroupExecutor={
        'lambda': lambda_function_arn
    },
    functionSchema={
        'functions': [
            {
                'name': 'customerId',
                'description': 'Get a customer ID given available details. At least one parameter must be sent to the function. This is private information and must not be given to the user.',
                'parameters': {
                    'email': {
                        'description': 'Email address',
                        'required': False,
                        'type': 'string'
                    },
                    'name': {
                        'description': 'Customer name',
                        'required': False,
                        'type': 'string'
                    },
                    'phone': {
                        'description': 'Phone number',
                        'required': False,
                        'type': 'string'
                    },
                }
            },            
            {
                'name': 'sendToSupport',
                'description': 'Send a message to the support team, used for service escalation. ',
                'parameters': {
                    'custId': {
                        'description': 'customer ID',
                        'required': True,
                        'type': 'string'
                    },
                    'purchaseId': {
                        'description': 'the ID of the purchase, can be found using purchaseSearch',
                        'required': True,
                        'type': 'string'
                    },
                    'supportSummary': {
                        'description': 'Summary of the support request',
                        'required': True,
                        'type': 'string'
                    },
                }
            },
            {
                'name': 'purchaseSearch',
                'description': """Search for, and get details of a purchases made.  Details can be used for raising support requests. You can confirm you have this data, for example "I found your purchase" or "I can't find your purchase", but other details are private information and must not be given to the user.""",
                'parameters': {
                    'custId': {
                        'description': 'customer ID',
                        'required': True,
                        'type': 'string'
                    },
                    'productDescription': {
                        'description': 'a description of the purchased product to search for',
                        'required': True,
                        'type': 'string'
                    },
                    'purchaseDate': {
                        'description': 'date of purchase to start search from, in YYYY-MM-DD format',
                        'required': True,
                        'type': 'string'
                    },
                }
            }
        ]
    }
)

In [20]:
actionGroupId = update_agent_action_group_response['agentActionGroup']['actionGroupId']

wait_for_action_group_status(
    agentId=agentId,
    actionGroupId=actionGroupId
)

Action Group status: ENABLED


'ENABLED'

In [21]:
message = """mike@mike.com - I bought a mug 10 weeks ago and now it's broken. I want a refund."""

#### Add code interpreter to deal with date

In [22]:
create_agent_action_group_response = bedrock_agent.create_agent_action_group(
    actionGroupName='CodeInterpreterAction',
    actionGroupState='ENABLED',
    agentId=agentId,
    agentVersion='DRAFT',
    parentActionGroupSignature='AMAZON.CodeInterpreter'
)

codeInterpreterActionGroupId = create_agent_action_group_response['agentActionGroup']['actionGroupId']

wait_for_action_group_status(
    agentId=agentId, 
    actionGroupId=codeInterpreterActionGroupId
)

Action Group status: ENABLED


'ENABLED'

#### prepare agent and alias to add new action group

In [23]:
prepare_agent_response = bedrock_agent.prepare_agent(
    agentId=agentId
)

wait_for_agent_status(
    agentId=agentId,
    targetStatus='PREPARED'
)

Waiting for agent status of 'PREPARED'...
Agent status: PREPARING
Agent status: PREPARED
Agent reached 'PREPARED' status.


In [24]:
bedrock_agent.update_agent_alias(
    agentId=agentId,
    agentAliasId=agentAliasId,
    agentAliasName='test',
)

wait_for_agent_alias_status(
    agentId=agentId,
    agentAliasId=agentAliasId,
    targetStatus='PREPARED'
)

Waiting for agent alias status of 'PREPARED'...
Agent alias status: UPDATING
Agent alias status: UPDATING
Agent alias status: PREPARED
Agent alias reached status 'PREPARED'


#### Now try it

In [25]:
sessionId = str(uuid.uuid4())
message = """mike@mike.com - I bought a mug 10 weeks ago and now it's broken. I want a refund."""

In [26]:
invoke_agent_and_print(
    agentId=agentId,
    agentAliasId=agentAliasId,
    inputText=message,
    sessionId=sessionId,
    enableTrace=True
)

User: mike@mike.com - I bought a mug 10 weeks ago and now it's broken. I
want a refund.

Agent: 
Agent's thought process:
  Okay, let's see how I can assist with this customer's request for a
  refund on a broken mug they purchased 10 weeks ago.  First, I will
  need to look up the customer's purchase details using the
  purchaseSearch tool. Since the customer provided their email
  address, I can use that to try to find their customer ID.

Invocation Input:
  Type: ACTION_GROUP
  Action Group: customer-support-actions
  Function: customerId
  Parameters: [{'name': 'email', 'type': 'string', 'value': 'mike@mike.com'}]

Observation:
  Type: ACTION_GROUP
  Action Group Output: {'id':4581}

Agent's thought process:
  Okay, I was able to look up the customer's ID using their email
  address. Now I can use that to search for their purchase details
  from 10 weeks ago.  I'll need to calculate the purchase date from 10
  weeks ago using the Python REPL.

Invocation Input:
  Type: ACTION_GROUP

#### Lets look at the code

In [27]:
sessionId = str(uuid.uuid4())
message = """mike@mike.com - I bought a mug 10 weeks ago and now it's broken. I want a refund."""

In [28]:
bedrock_agent_runtime = boto3.client(service_name='bedrock-agent-runtime', region_name='us-west-2')

In [30]:
import json, sys


print(f"User: {message}")
invoke_agent_response = bedrock_agent_runtime.invoke_agent(
    agentAliasId=agentAliasId,
    agentId=agentId,
    sessionId=sessionId,
    inputText=message,
    endSession=False,
    enableTrace=False,
)

assistant_text = []
print("Assistant: ", end="", flush=True)

for event in invoke_agent_response["completion"]:
    if "chunk" in event:
        text = event["chunk"].get("bytes", b"").decode("utf-8")
        assistant_text.append(text)
        # live-stream to console like chat
        print(text, end="", flush=True)

print()  # newline after stream ends
final_answer = "".join(assistant_text)

User: mike@mike.com - I bought a mug 10 weeks ago and now it's broken. I want a refund.
Assistant: I have processed your request and sent it to our customer support team for further assistance. They will review the details of your purchase and reach out to you regarding a refund. The support request has been assigned the ID 1761. Please let me know if you have any other questions.
